In [1]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import f_classif
import tensorflow as tf

2026-07-20 17:18:07.676379: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-20 17:18:07.677214: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-20 17:18:07.679404: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-20 17:18:07.685816: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-20 17:18:07.696329: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 

In [2]:
raw_df = pd.read_csv('../Data/Audiobooks_data.csv')
raw_df.describe()

,ID,Book length (minutes)_overall,Book length (minutes)_avg,Price_overall,Price_avg,Review,Review 10/10,Completion,Minutes listened,Support Request,Last visited minus purchase date,Targets
count,14084.000000,14084.000000,14084.000000,14084.000000,14084.000000,14084.000000,14084.000000,14084.000000,14084.000000,14084.000000,14084.000000,14084.000000
mean,16772.491551,1591.281685,1678.608634,7.103791,7.543805,0.160750,8.909795,0.125659,189.888983,0.070222,61.935033,0.158833
std,9691.807248,504.340663,654.838599,4.931673,5.560129,0.367313,0.643406,0.241206,371.084010,0.472157,88.207634,0.365533
min,2.000000,216.000000,216.000000,3.860000,3.860000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,8368.000000,1188.000000,1188.000000,5.330000,5.330000,0.000000,8.910000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,16711.500000,1620.000000,1620.000000,5.950000,6.070000,0.000000,8.910000,0.000000,0.000000,0.000000,11.000000,0.000000
75%,25187.250000,2160.000000,2160.000000,8.000000,8.000000,0.000000,8.910000,0.130000,194.400000,0.000000,105.000000,0.000000
max,33683.000000,2160.000000,7020.000000,130.940000,130.940000,1.000000,10.000000,1.000000,2160.000000,30.000000,464.000000,1.000000


Data Cleaning

In [3]:
raw_df.dropna()  #although there is no null value
raw_df.shape
raw_df = raw_df.drop("ID", axis=1)

In [4]:
raw_df["Targets"].value_counts()

Targets
0    11847
1     2237
Name: count, dtype: int64

In [5]:
#Balance the dataset --> Equal number of 1s and 0s --> to ensure unbiased data for modeling

majority = raw_df[raw_df["Targets"] == 0]
minority = raw_df[raw_df["Targets"] == 1]

majority_downsampled = resample(
    majority,
    replace=False,
    n_samples=len(minority),
    random_state=42
)

df_balanced = pd.concat([majority_downsampled, minority])
df_balanced

,Book length (minutes)_overall,Book length (minutes)_avg,Price_overall,Price_avg,Review,Review 10/10,Completion,Minutes listened,Support Request,Last visited minus purchase date,Targets
9786,1188.0,1188,5.33,5.33,0,8.91,0.00,0.00,0,0,0
1204,1188.0,1188,5.33,5.33,0,8.91,0.00,0.00,0,0,0
4846,1620.0,1620,7.47,7.47,0,8.91,0.02,32.40,0,296,0
11647,1188.0,1188,5.33,5.33,0,8.91,0.14,166.32,0,2,0
3567,1620.0,1620,5.87,5.87,0,8.91,0.00,0.00,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
14056,1476.0,4428,5.33,16.00,0,8.91,0.00,0.00,0,0,1
14059,2160.0,2160,5.33,5.33,0,8.91,0.00,0.00,0,12,1
14070,2160.0,2160,5.33,5.33,1,10.00,0.00,0.00,0,6,1
14076,1674.0,3348,7.99,15.99,0,8.91,0.00,0.00,0,0,1


In [6]:
df_balanced["Targets"].value_counts() #This time it will output the same number because the data is balanced now

Targets
0    2237
1    2237
Name: count, dtype: int64

In [7]:
df_shuffled = resample(df_balanced, replace=False, random_state=42)
bal_shuffled_df = df_shuffled.reset_index(drop=True)
bal_shuffled_df.head(100)

,Book length (minutes)_overall,Book length (minutes)_avg,Price_overall,Price_avg,Review,Review 10/10,Completion,Minutes listened,Support Request,Last visited minus purchase date,Targets
0,702.0,1404,8.36,16.72,0,8.91,0.00,0.0,0,0,1
1,2160.0,2160,5.33,5.33,0,8.91,0.00,0.0,0,115,1
2,2160.0,2160,7.99,7.99,0,8.91,0.00,0.0,0,0,0
3,1620.0,1620,5.33,5.33,0,8.91,0.00,0.0,0,0,0
4,1620.0,1620,8.00,8.00,0,8.91,0.00,0.0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...
95,1620.0,1620,10.13,10.13,0,8.91,0.00,0.0,0,0,0
96,2160.0,2160,5.33,5.33,0,8.91,0.00,0.0,0,12,1
97,1188.0,1188,5.33,5.33,0,8.91,0.00,0.0,0,0,1
98,1620.0,1620,6.40,6.40,1,9.00,0.48,777.6,0,245,0


In [8]:
#checking
bal_shuffled_df['Targets'].sum()

2237

In [9]:
y = bal_shuffled_df['Targets']
X = bal_shuffled_df.iloc[:, :-1]
X

,Book length (minutes)_overall,Book length (minutes)_avg,Price_overall,Price_avg,Review,Review 10/10,Completion,Minutes listened,Support Request,Last visited minus purchase date
0,702.0,1404,8.36,16.72,0,8.91,0.0,0.0,0,0
1,2160.0,2160,5.33,5.33,0,8.91,0.0,0.0,0,115
2,2160.0,2160,7.99,7.99,0,8.91,0.0,0.0,0,0
3,1620.0,1620,5.33,5.33,0,8.91,0.0,0.0,0,0
4,1620.0,1620,8.00,8.00,0,8.91,0.0,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...
4469,1188.0,1188,5.33,5.33,0,8.91,0.0,0.0,0,153
4470,1080.0,1080,7.72,7.72,0,8.91,0.0,0.0,0,0
4471,1188.0,4752,5.73,22.91,0,8.91,0.0,0.0,0,282
4472,1620.0,1620,5.61,5.61,0,10.00,0.0,0.0,0,98


In [10]:
X_scaled = pd.DataFrame(preprocessing.scale(X))

In [11]:
X_train, X_temp, y_train, y_temp = train_test_split(X_scaled, y, test_size=0.3, random_state=10)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.5, random_state=10)

In [12]:
print(X_train.shape)
print(X_test.shape)
print(X_val.shape)
print(X_train.shape[0]+ X_test.shape[0] + X_val.shape[0])

(3131, 10)
(671, 10)
(672, 10)
4474


In [13]:
X_train

,0,1,2,3,4,5,6,7,8,9
3117,0.105261,-0.255813,0.516311,0.212191,-0.452362,-0.01247,-0.392052,-0.387206,4.446612,1.676762
1284,-0.764398,-0.752238,-0.238760,-0.383294,-0.452362,-0.01247,-0.392052,-0.387206,-0.200466,-0.759914
736,-0.764398,-0.752238,-0.340508,-0.463537,-0.452362,-0.01247,-0.392052,-0.387206,-0.200466,-0.759914
3946,-0.764398,-0.752238,0.516311,0.212191,-0.452362,-0.01247,-0.392052,-0.387206,-0.200466,1.213147
252,1.192335,0.364718,-0.290527,-0.424120,-0.452362,-0.01247,-0.392052,-0.387206,-0.200466,-0.253171
...,...,...,...,...,...,...,...,...,...,...
2009,0.177733,3.591479,0.066481,1.998646,-0.452362,-0.01247,-0.392052,-0.387206,-0.200466,1.170020
1180,0.105261,-0.255813,-0.340508,-0.463537,-0.452362,-0.01247,-0.392052,-0.387206,-0.200466,-0.759914
3441,-0.764398,-0.752238,-0.340508,-0.463537,-0.452362,-0.01247,-0.392052,-0.387206,-0.200466,-0.759914
1344,0.105261,-0.255813,0.448480,0.158696,-0.452362,-0.01247,-0.392052,-0.387206,-0.200466,-0.759914


In [14]:
f_values, p_values = f_classif(X_train, y_train)

# Put results in a table
feature_summary = pd.DataFrame({
    'Feature': np.arange(X_train.shape[1]),
    'F_value': f_values,
    'p_value': p_values
})

# Round for readability
feature_summary = feature_summary.round(3)
feature_summary

,Feature,F_value,p_value
0,0,27.899,0.000
1,1,222.665,0.000
2,2,0.002,0.967
3,3,135.680,0.000
4,4,0.329,0.567
5,5,1.209,0.272
6,6,597.994,0.000
7,7,589.957,0.000
8,8,3.495,0.062
9,9,52.904,0.000


In [15]:
col_to_delete = []
for i in range(len(feature_summary['p_value'])):
    if feature_summary['p_value'][i] >0.05:
        col_to_delete.append(i)


In [16]:
X_train

,0,1,2,3,4,5,6,7,8,9
3117,0.105261,-0.255813,0.516311,0.212191,-0.452362,-0.01247,-0.392052,-0.387206,4.446612,1.676762
1284,-0.764398,-0.752238,-0.238760,-0.383294,-0.452362,-0.01247,-0.392052,-0.387206,-0.200466,-0.759914
736,-0.764398,-0.752238,-0.340508,-0.463537,-0.452362,-0.01247,-0.392052,-0.387206,-0.200466,-0.759914
3946,-0.764398,-0.752238,0.516311,0.212191,-0.452362,-0.01247,-0.392052,-0.387206,-0.200466,1.213147
252,1.192335,0.364718,-0.290527,-0.424120,-0.452362,-0.01247,-0.392052,-0.387206,-0.200466,-0.253171
...,...,...,...,...,...,...,...,...,...,...
2009,0.177733,3.591479,0.066481,1.998646,-0.452362,-0.01247,-0.392052,-0.387206,-0.200466,1.170020
1180,0.105261,-0.255813,-0.340508,-0.463537,-0.452362,-0.01247,-0.392052,-0.387206,-0.200466,-0.759914
3441,-0.764398,-0.752238,-0.340508,-0.463537,-0.452362,-0.01247,-0.392052,-0.387206,-0.200466,-0.759914
1344,0.105261,-0.255813,0.448480,0.158696,-0.452362,-0.01247,-0.392052,-0.387206,-0.200466,-0.759914


In [17]:
X_train = X_train.drop(columns=col_to_delete)
X_val = X_val.drop(columns=col_to_delete)
X_test = X_test.drop(columns=col_to_delete)
X_train

,0,1,3,6,7,9
3117,0.105261,-0.255813,0.212191,-0.392052,-0.387206,1.676762
1284,-0.764398,-0.752238,-0.383294,-0.392052,-0.387206,-0.759914
736,-0.764398,-0.752238,-0.463537,-0.392052,-0.387206,-0.759914
3946,-0.764398,-0.752238,0.212191,-0.392052,-0.387206,1.213147
252,1.192335,0.364718,-0.424120,-0.392052,-0.387206,-0.253171
...,...,...,...,...,...,...
2009,0.177733,3.591479,1.998646,-0.392052,-0.387206,1.170020
1180,0.105261,-0.255813,-0.463537,-0.392052,-0.387206,-0.759914
3441,-0.764398,-0.752238,-0.463537,-0.392052,-0.387206,-0.759914
1344,0.105261,-0.255813,0.158696,-0.392052,-0.387206,-0.759914


In [18]:
# #Balance the dataset --> Equal number of 1s and 0s --> to ensure unbiased data for modeling
# num_one_target = int(sum(unscaled_target))
# zero_targets_counter = 0
# indices_to_remove = []
# for i in range(unscaled_target.shape[0]):
#     if unscaled_target[i] == 0:
#         zero_targets_counter += 1
#         if zero_targets_counter > num_one_target:
#             indices_to_remove.append(i)
# targets_equal_priors = np.delete(unscaled_target, indices_to_remove, axis = 0)
# unscaled_inputs_equal_prior = np.delete(unscaled_inputs_all, indices_to_remove, axis = 0)
# print(len(targets_equal_priors))
# print(int(sum(unscaled_target)))

In [19]:
# scaled_inputs = preprocessing.scale(unscaled_inputs_equal_prior)

In [20]:
# shuffled_indices = np.arange(scaled_inputs.shape[0])
# np.random.shuffle(shuffled_indices)
# shuffled_inputs = scaled_inputs[shuffled_indices]
# shuffled_targets = targets_equal_priors[shuffled_indices]

In [21]:
#for model training
np.savez('../Data/Audiobooks_data_train', inputs=X_train, outputs=y_train)
np.savez('../Data/Audiobooks_data_validation', inputs=X_val, outputs=y_val)
np.savez('../Data/Audiobooks_data_test', inputs=X_test, outputs=y_test)

#for dashboards
train_df = X_train
train_df['Targets'] = y_train
train_df.to_csv("../Data/Audiobooks_data_train.csv", index=False)

val_df = X_val
val_df['Targets'] = y_val
val_df.to_csv("../Data/Audiobooks_data_validation.csv", index=False)

test_df = X_test
test_df['Targets'] = y_test
test_df.to_csv("../Data/Audiobooks_data_test.csv", index=False)